In [ ]:
from musicProcessing import MusicProcessingProvider as mp
from alg import Alg
from alg_bins import AlgBins
from alg_binary import AlgBinary
from individual import Individual
from individual_bins import Individual_bins
from individual_binary import Individual_binary

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import multiprocessing

In [ ]:
# mp params
path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/babyslakh_16k/Track00001/mix.wav"
duration = None
nfft = 2048
win_length = 2048
hop_length = 1024
window_type = "hann"

music_processor = mp(path, duration, nfft, hop_length, win_length, window_type)

In [ ]:
# stable alg params
num_generations = 100
population_size = 200
sharing_radius = 0.5
mutation_probability = 0.15
crossover_probability = 0.7
population_initialization_method = "random" # "random" or "elite"
selection_method = "tournament" # "tournament" or "roulette"
Individual_bin_count = 20

In [ ]:
similarity_metrics = ["jaccard", "cosine", "fenotype"]
crossover_methods = ["one_point", "two_point"]
random_seeds = np.arange(20)

results_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/results/exp1/"
os.makedirs(results_path, exist_ok=True)

In [ ]:
global_df = pd.DataFrame(columns=["similarity_metric", "crossover_method", "best_end_fitness", "best_end_fitness_std", "best_end_shared_fitness", "best_end_shared_fitness_std", "avg_end_similarity", "avg_end_similarity_std"])

In [ ]:
def run_similarity_metric(similarity_metric):
    print(f"--- Starting worker process for metric: {similarity_metric} ---")

    # Create a local DataFrame for this specific process
    local_df = pd.DataFrame(
        columns=[
            "similarity_metric",
            "crossover_method",
            "best_end_fitness",
            "best_end_fitness_std",
            "best_end_shared_fitness",
            "best_end_shared_fitness_std",
            "avg_end_similarity",
            "avg_end_similarity_std",
        ]
    )

    for crossover_method in crossover_methods:
        best_fitness = np.zeros((num_generations, len(random_seeds)))
        best_shared_fitness = np.zeros((num_generations, len(random_seeds)))
        avg_similarity = np.zeros((num_generations, len(random_seeds)))

        for idx, random_seed in enumerate(random_seeds):
            print(
                f"[{similarity_metric}] Running crossover: {crossover_method}, seed: {random_seed}"
            )

            # Initialize and run algorithm
            alg = Alg(
                music_processor,
                num_generations,
                population_size,
                sharing_radius,
                mutation_probability,
                crossover_probability,
                population_initialization_method,
                selection_method,
                similarity_metric,
                crossover_method,
                random_seed,
            )
            alg.run()

            # Using idx instead of random_seeds.index() is safer and faster
            best_fitness[:, idx] = alg.best_fitness
            best_shared_fitness[:, idx] = alg.best_shared_fitness
            avg_similarity[:, idx] = alg.avg_similarity

        # --- Plotting section (unchanged) ---
        plt.figure(figsize=(14, 8))
        plt.subplot(2, 2, 1)
        plt.plot(avg_similarity.mean(axis=1), label="Similarity")
        plt.xlabel("Generation")
        plt.ylabel("Similarity")
        plt.title("Population Similarity Over Generations")
        plt.legend()

        plt.subplot(2, 2, 2)
        plt.plot(best_shared_fitness.mean(axis=1), label="Best")
        plt.xlabel("Generation")
        plt.ylabel("Shared Fitness")
        plt.title("Best Shared Fitness Over Generations")
        plt.legend()

        plt.subplot(2, 2, 3)
        plt.plot(best_fitness.mean(axis=1), label="Best")
        plt.xlabel("Generation")
        plt.ylabel("Fitness")
        plt.title("Best Fitness Over Generations")
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            os.path.join(
                results_path,
                f"similarity_{similarity_metric}_crossover_{crossover_method}.png",
            )
        )
        plt.close()

        # --- Append to local DataFrame ---
        row = pd.DataFrame(
            [
                {
                    "similarity_metric": similarity_metric,
                    "crossover_method": crossover_method,
                    "best_end_fitness": best_fitness[-1, :].mean(),
                    "best_end_fitness_std": best_fitness[-1, :].std(),
                    "best_end_shared_fitness": best_shared_fitness[
                        -1, :
                    ].mean(),
                    "best_end_shared_fitness_std": best_shared_fitness[
                        -1, :
                    ].std(),
                    "avg_end_similarity": avg_similarity[-1, :].mean(),
                    "avg_end_similarity_std": avg_similarity[-1, :].std(),
                }
            ]
        )
        local_df = pd.concat([local_df, row], ignore_index=True)

    return local_df

In [ ]:
# 2. Main execution guard (Crucial for multiprocessing in Python)
if __name__ == "__main__":
    # Create the directory if it doesn't exist yet
    os.makedirs(results_path, exist_ok=True)

    # Launch a pool of workers. It defaults to the number of CPU cores,
    # but since you have 3 metrics, it will spin up 3 concurrent jobs perfectly.
    with multiprocessing.Pool() as pool:
        # pool.map runs 'run_similarity_metric' for each item in the list concurrently
        results_dfs = pool.map(run_similarity_metric, similarity_metrics)

    # 3. Combine results back in the main process
    global_df = pd.concat(results_dfs, ignore_index=True)
    global_df.to_csv(os.path.join(results_path, "results.csv"), index=False)
    print("All experiments finished! Data saved to results.csv")